# SmolLM3-3B Adventure-Command Fine-Tuning
LoRA fine-tune for NPC cognitive architecture Layer 3.

**Instructions:** Runtime → Change runtime type → GPU (A100 or T4), then Run All.
Upload `command_training_shuffled.jsonl` when prompted.

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="HuggingFaceTB/SmolLM3-3B",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# Upload training data
from google.colab import files
print("Upload command_training_shuffled.jsonl (2.6MB)")
uploaded = files.upload()

In [ ]:
# Load and preprocess dataset
import json
from datasets import Dataset

examples = []
with open("command_training_shuffled.jsonl") as f:
    for line in f:
        examples.append(json.loads(line))

dataset = Dataset.from_list(examples)
print(f"Loaded {len(dataset)} examples")

def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )}

dataset = dataset.map(to_text, remove_columns=["messages"])
print(f"Dataset ready: {len(dataset)} examples")

In [ ]:
# Train
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="smollm3-command-lora",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=1.5e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=2,
        report_to="none",
        optim="adamw_8bit",
    ),
    max_seq_length=2048,
)

trainer.train()

In [ ]:
# Test
FastLanguageModel.for_inference(model)
model.config._attn_implementation = "eager"
if hasattr(model, "base_model"):
    model.base_model.model.config._attn_implementation = "eager"
    for layer in model.base_model.model.model.layers:
        layer.self_attn.config._attn_implementation = "eager"

def test_model(prompt_text):
    messages = [
        {"role": "system", "content": "You are the mind of a simulated being. You receive your state and perceptions, think about your situation, then choose one action. Think inside <think></think> tags, then output exactly one command."},
        {"role": "user", "content": prompt_text},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    attention_mask = torch.ones_like(inputs)
    outputs = model.generate(input_ids=inputs, attention_mask=attention_mask, max_new_tokens=200, temperature=0.4, do_sample=True, top_p=0.9)
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("=== Hungry guard ===")
print(test_model("BEING: guard\nDRIVES: energy=30 hunger=85 social=40 safety=70\nMOOD: fatigue 0.5\nLOCATION: guard_post\nDOING: standing watch\n\nYou see:\n  (no one)\nYou hear:\n  (nothing)\nNearby objects:\n  Weapon Rack (working)\n\nRecent: Nothing notable.\nGoal: finish the watch\nBeliefs: none\nLast thought: Stomach won't stop growling.\n\nAvailable places: guard_post, inn, market, town_square, home_north\n\nThink about your situation, then choose ONE action."))

In [ ]:
# Export GGUF + save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained_merged("smollm3-command-merged", tokenizer, save_method="merged_16bit")
model.save_pretrained_gguf("smollm3-command-gguf", tokenizer, quantization_method="q4_k_m")

import glob, shutil, os
ggufs = glob.glob("/content/**/SmolLM3*.Q4_K_M.gguf", recursive=True)
if ggufs:
    src = ggufs[0]
    dst = "/content/drive/MyDrive/SmolLM3-3B.Q4_K_M.gguf"
    print(f"Copying {src} ({os.path.getsize(src)/1e9:.2f} GB) to Drive...")
    shutil.copy2(src, dst)
    print(f"DONE! Saved to Google Drive: My Drive/SmolLM3-3B.Q4_K_M.gguf")
else:
    print("ERROR: GGUF not found!")
    !find /content -name '*.gguf' -ls

# LoRA backup
model.save_pretrained("smollm3-command-lora-final")
tokenizer.save_pretrained("smollm3-command-lora-final")
!zip -qr /content/drive/MyDrive/smollm3-command-lora.zip smollm3-command-lora-final/
print("LoRA adapter also saved to Drive.")